# TFG — Predicción de lesiones en corredores de resistencia
## Un enfoque basado en el análisis de la carga de entrenamiento

**Grado en Business Intelligence / ADE**  
**Curso 2024–2025**

---

### Estructura del notebook

| Bloque | Contenido |
|--------|-----------|
| 1 | Carga de datos y análisis exploratorio (EDA) |
| 2 | Ingeniería de variables |
| 3 | Preparación para modelado |
| 4 | Modelos predictivos *(próximamente)* |
| 5 | Evaluación y comparación *(próximamente)* |

---

> **Instrucciones:** Ejecuta las celdas en orden con `Shift+Enter`.  
> Los CSV `timeseries (daily).csv` y `timeseries (weekly).csv` deben estar en la misma carpeta que este notebook.

---
## Bloque 1 — Carga de datos y análisis exploratorio (EDA)

En este bloque cargamos los dos datasets (diario y semanal), verificamos su estructura,
analizamos la variable objetivo `injury` y exploramos las principales variables de entrenamiento.

**Hallazgos clave que anticipamos:**
- Desbalanceo severo de clases (~1.3% de positivos)
- Sin valores nulos en ninguno de los dos datasets
- Las variables de carga subjetiva son más diferenciadoras que el volumen puro

In [ ]:
# ── Imports y configuración global ──────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#2196F3', '#F44336']

# Ruta automática a la carpeta del notebook
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
print("Directorio de trabajo:", BASE_DIR)

In [ ]:
# ── Carga de datos ───────────────────────────────────────────────────────────
PATH_DAILY  = os.path.join(BASE_DIR, "timeseries (daily).csv")
PATH_WEEKLY = os.path.join(BASE_DIR, "timeseries (weekly).csv")

df_daily  = pd.read_csv(PATH_DAILY)
df_weekly = pd.read_csv(PATH_WEEKLY)

df_daily  = df_daily.sort_values(['Athlete ID', 'Date']).reset_index(drop=True)
df_weekly = df_weekly.sort_values(['Athlete ID', 'Date']).reset_index(drop=True)

print("=" * 55)
print("RESUMEN DE LOS DATASETS")
print("=" * 55)
print(f"  Daily  — filas: {len(df_daily):,}  |  columnas: {df_daily.shape[1]}")
print(f"  Weekly — filas: {len(df_weekly):,}  |  columnas: {df_weekly.shape[1]}")
print(f"  Atletas unicos (daily):  {df_daily['Athlete ID'].nunique()}")
print(f"  Atletas unicos (weekly): {df_weekly['Athlete ID'].nunique()}")
print(f"  Rango temporal: {df_daily['Date'].min()} - {df_daily['Date'].max()}")
print(f"  Valores nulos — daily:  {df_daily.isnull().sum().sum()}")
print(f"  Valores nulos — weekly: {df_weekly.isnull().sum().sum()}")

### 1.1 Variable objetivo — desbalanceo de clases

In [ ]:
# ── Distribución de injury ────────────────────────────────────────────────────
for label, df in [("Daily", df_daily), ("Weekly", df_weekly)]:
    vc   = df['injury'].value_counts()
    rate = df['injury'].mean() * 100
    print(f"[{label}]  injury=0: {vc[0]:,}  |  injury=1: {vc[1]:,}  |  "
          f"Tasa: {rate:.2f}%  |  Ratio: 1:{vc[0]/vc[1]:.0f}")

# Figura 1
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (label, df) in zip(axes, [("Diario", df_daily), ("Semanal", df_weekly)]):
    counts = df['injury'].value_counts().sort_index()
    bars   = ax.bar(['Sin lesion', 'Lesion'], counts.values,
                    color=PALETTE, edgecolor='white', width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', fontsize=10)
    ax.set_title(f'Distribucion de clases — {label}')
    ax.set_ylabel('Numero de registros')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.suptitle('Desbalanceo de la variable objetivo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig01_desbalanceo_clases.png'), bbox_inches='tight')
plt.show()

### 1.2 Análisis por atleta

In [ ]:
# ── Heterogeneidad entre atletas ─────────────────────────────────────────────
obs_por_atleta      = df_weekly.groupby('Athlete ID').size()
lesiones_por_atleta = df_weekly.groupby('Athlete ID')['injury'].sum()

print(f"Observaciones semanales — Media: {obs_por_atleta.mean():.0f} | "
      f"Min: {obs_por_atleta.min()} | Max: {obs_por_atleta.max()}")
print(f"Lesiones por atleta    — Media: {lesiones_por_atleta.mean():.1f} | "
      f"Max: {lesiones_por_atleta.max()}")
print(f"Atletas con 0 lesiones: {(lesiones_por_atleta==0).sum()}")
print(f"Atletas con 1+ lesion:  {(lesiones_por_atleta>=1).sum()}")

# Figura 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(obs_por_atleta)),
            obs_por_atleta.sort_values(ascending=False).values,
            color='#2196F3', alpha=0.8)
axes[0].set_title('Semanas de seguimiento por atleta')
axes[0].set_xlabel('Atleta (ordenado)')
axes[0].set_ylabel('Numero de semanas')

axes[1].bar(range(len(lesiones_por_atleta)),
            lesiones_por_atleta.sort_values(ascending=False).values,
            color='#F44336', alpha=0.8)
axes[1].set_title('Semanas con lesion por atleta')
axes[1].set_xlabel('Atleta (ordenado)')
axes[1].set_ylabel('Semanas con lesion')

plt.suptitle('Heterogeneidad entre atletas', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig02_atletas.png'), bbox_inches='tight')
plt.show()

### 1.3 Cálculo del ACWR y análisis por clase

In [ ]:
# ── ACWR ─────────────────────────────────────────────────────────────────────
df_weekly['chronic_kms'] = (
    df_weekly['total kms'] + df_weekly['total kms.1'] + df_weekly['total kms.2']
) / 3

df_weekly['ACWR'] = np.where(
    df_weekly['chronic_kms'] > 0,
    df_weekly['total kms'] / df_weekly['chronic_kms'],
    np.nan
)
df_weekly['ACWR'] = df_weekly['ACWR'].clip(upper=3.0)

acwr_by_injury = df_weekly.groupby('injury')['ACWR'].mean()
print(f"ACWR medio sin lesion: {acwr_by_injury[0]:.3f}")
print(f"ACWR medio con lesion: {acwr_by_injury[1]:.3f}")
print(f"Diferencia relativa:   "
      f"{(acwr_by_injury[1]-acwr_by_injury[0])/acwr_by_injury[0]*100:+.1f}%")

# Comparacion variables clave
features_analisis = {
    'total kms':  'Kilometros totales',
    'ACWR':       'ACWR',
    'avg exertion': 'Esfuerzo percibido (media)',
    'avg recovery': 'Recuperacion percibida (media)',
    'nr. rest days': 'Dias de descanso',
    'nr. tough sessions (effort in Z5, T1 or T2)': 'Sesiones duras',
    'max km one day': 'Maximo km en un dia',
    'total km Z5-T1-T2': 'km alta intensidad',
}
print(f"\n{'Variable':<40} {'Sin lesion':>12} {'Lesion':>12} {'D%':>8}")
print("-" * 75)
for col, label in features_analisis.items():
    m0   = df_weekly[df_weekly['injury']==0][col].mean()
    m1   = df_weekly[df_weekly['injury']==1][col].mean()
    diff = (m1 - m0) / abs(m0) * 100 if m0 != 0 else np.nan
    print(f"{label:<40} {m0:>12.3f} {m1:>12.3f} {diff:>+8.1f}%")

In [ ]:
# ── Figura 3: Boxplots por clase ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
valid_features = [(col, lbl) for col, lbl in features_analisis.items()
                  if col in df_weekly.columns]
for ax, (col, label) in zip(axes, valid_features):
    data_0 = df_weekly[df_weekly['injury']==0][col].dropna()
    data_1 = df_weekly[df_weekly['injury']==1][col].dropna()
    bp = ax.boxplot([data_0, data_1], patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3),
                    whiskerprops=dict(linewidth=1.2),
                    boxprops=dict(linewidth=1.2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Sin lesion', 'Lesion'], fontsize=9)
    ax.set_title(label, fontsize=10)
plt.suptitle('Variables clave por clase de lesion', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig03_boxplots_variables.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Figura 4: ACWR por clase y tasa de lesion ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for val, label, color in [(0, 'Sin lesion', '#2196F3'), (1, 'Lesion', '#F44336')]:
    data = df_weekly[df_weekly['injury']==val]['ACWR'].dropna()
    axes[0].hist(data, bins=50, alpha=0.6, color=color,
                 label=f'{label} (n={len(data):,})', density=True)
axes[0].axvline(1.0, color='gray', linestyle='--', linewidth=1, label='ACWR=1.0')
axes[0].axvline(1.5, color='orange', linestyle='--', linewidth=1, label='ACWR=1.5')
axes[0].set_xlabel('ACWR'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribucion del ACWR por clase'); axes[0].legend(fontsize=9)

bins = [0, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.01]
labels_bins = ['<0.5','0.5-0.8','0.8-1.0','1.0-1.2','1.2-1.5','1.5-2.0','>2.0']
df_weekly['ACWR_bin'] = pd.cut(df_weekly['ACWR'], bins=bins, labels=labels_bins)
injury_rate_by_acwr = df_weekly.groupby('ACWR_bin', observed=True)['injury'].mean() * 100
axes[1].bar(injury_rate_by_acwr.index, injury_rate_by_acwr.values,
            color='#E91E63', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Rango de ACWR'); axes[1].set_ylabel('Tasa de lesion (%)')
axes[1].set_title('Tasa de lesion por rango de ACWR')
axes[1].yaxis.set_major_formatter(PercentFormatter(decimals=2))
plt.suptitle('Analisis del ACWR', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig04_ACWR_analisis.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Figura 5: Correlaciones con injury ───────────────────────────────────────
base_numeric_cols = [
    c for c in df_weekly.columns
    if '.' not in c
    and c not in ['Athlete ID', 'Date', 'injury', 'ACWR_bin', 'chronic_kms']
    and df_weekly[c].dtype in [np.float64, np.int64]
]
corr_with_injury = (
    df_weekly[base_numeric_cols + ['ACWR', 'injury']]
    .corr()['injury'].drop('injury')
    .sort_values(key=abs, ascending=False)
)
print("Top 10 correlaciones con injury:")
print(corr_with_injury.head(10).round(4).to_string())

top_corr    = corr_with_injury.head(15)
colors_corr = ['#F44336' if v > 0 else '#2196F3' for v in top_corr.values]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_corr)), top_corr.values, color=colors_corr, alpha=0.8)
ax.set_yticks(range(len(top_corr))); ax.set_yticklabels(top_corr.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlacion de Pearson con injury')
ax.set_title('Top 15 variables mas correlacionadas con la lesion')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig05_correlaciones.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Figura 6: Evolucion temporal ─────────────────────────────────────────────
df_weekly['date_bin'] = pd.cut(df_weekly['Date'], bins=27)
temporal = df_weekly.groupby('date_bin', observed=True)['injury'].mean() * 100
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(temporal)), temporal.values,
        color='#F44336', linewidth=2, marker='o', markersize=4)
ax.fill_between(range(len(temporal)), temporal.values, alpha=0.15, color='#F44336')
ax.set_xlabel('Periodo del estudio (progresivo)')
ax.set_ylabel('Tasa de lesion (%)')
ax.set_title('Evolucion temporal de la tasa de lesion')
ax.yaxis.set_major_formatter(PercentFormatter(decimals=2))
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig06_evolucion_temporal.png'), bbox_inches='tight')
plt.show()
print("Bloque 1 completado.")

---
## Bloque 2 — Ingeniería de variables

Construimos 8 variables nuevas a partir de los datos existentes:

| Variable | Definición | Justificación |
|----------|-----------|---------------|
| `ACWR` | Carga aguda / carga crónica (km) | Nakaoka et al. (2021) |
| `ACWR_exertion` | ACWR sobre esfuerzo percibido | Jones et al. (2017) |
| `monotony` | Media esfuerzo / variabilidad esfuerzo | Matos et al. (2020) |
| `strain` | Carga total × monotonía | Matos et al. (2020) |
| `km_change_w0_w1` | Cambio relativo km semana 0 vs 1 | Frandsen et al. (2025) |
| `km_change_w1_w2` | Cambio relativo km semana 1 vs 2 | Frandsen et al. (2025) |
| `recovery_trend` | Diferencia recuperación semana 0 vs 1 | — |
| `exertion_trend` | Diferencia esfuerzo semana 0 vs 1 | — |

In [ ]:
# ── ACWR sobre kilometros (ya calculado en Bloque 1) ─────────────────────────
# Ya disponible en df_weekly['ACWR']

# ── ACWR sobre esfuerzo percibido ────────────────────────────────────────────
df_weekly['chronic_exertion'] = (
    df_weekly['avg exertion'] + df_weekly['avg exertion.1'] + df_weekly['avg exertion.2']
) / 3
df_weekly['ACWR_exertion'] = np.where(
    df_weekly['chronic_exertion'] > 0,
    df_weekly['avg exertion'] / df_weekly['chronic_exertion'],
    np.nan
)
df_weekly['ACWR_exertion'] = df_weekly['ACWR_exertion'].clip(upper=3.0)

# ── Training Monotony ─────────────────────────────────────────────────────────
df_weekly['exertion_range'] = df_weekly['max exertion'] - df_weekly['min exertion']
df_weekly['monotony'] = np.where(
    df_weekly['exertion_range'] > 0,
    df_weekly['avg exertion'] / (df_weekly['exertion_range'] / 2),
    np.nan
)
df_weekly['monotony'] = df_weekly['monotony'].clip(upper=10.0)

# ── Training Strain ───────────────────────────────────────────────────────────
df_weekly['weekly_load'] = df_weekly['total kms'] * df_weekly['avg exertion']
df_weekly['strain'] = df_weekly['weekly_load'] * df_weekly['monotony']

# ── Ratios de cambio semanal ──────────────────────────────────────────────────
df_weekly['km_change_w0_w1'] = np.where(
    df_weekly['total kms.1'] > 0,
    (df_weekly['total kms'] - df_weekly['total kms.1']) / df_weekly['total kms.1'],
    np.nan
)
df_weekly['km_change_w0_w1'] = df_weekly['km_change_w0_w1'].clip(-1, 2)

df_weekly['km_change_w1_w2'] = np.where(
    df_weekly['total kms.2'] > 0,
    (df_weekly['total kms.1'] - df_weekly['total kms.2']) / df_weekly['total kms.2'],
    np.nan
)
df_weekly['km_change_w1_w2'] = df_weekly['km_change_w1_w2'].clip(-1, 2)

# ── Tendencias ────────────────────────────────────────────────────────────────
df_weekly['recovery_trend'] = df_weekly['avg recovery'] - df_weekly['avg recovery.1']
df_weekly['exertion_trend']  = df_weekly['avg exertion'] - df_weekly['avg exertion.1']

print("Variables derivadas creadas correctamente.")

In [ ]:
# ── Resumen comparativo por clase ────────────────────────────────────────────
new_vars = ['ACWR','ACWR_exertion','monotony','strain',
            'km_change_w0_w1','km_change_w1_w2','recovery_trend','exertion_trend']

print(f"{'Variable':<22} {'Sin lesion':>12} {'Lesion':>12} {'D%':>8}")
print("-" * 58)
for v in new_vars:
    m0   = df_weekly[df_weekly['injury']==0][v].mean()
    m1   = df_weekly[df_weekly['injury']==1][v].mean()
    diff = (m1 - m0) / abs(m0) * 100 if m0 != 0 else np.nan
    print(f"{v:<22} {m0:>12.4f} {m1:>12.4f} {diff:>+8.1f}%")

In [ ]:
# ── Figura 7: Boxplots variables derivadas ───────────────────────────────────
new_vars_labels = {
    'ACWR':             'ACWR (km)',
    'ACWR_exertion':    'ACWR (esfuerzo)',
    'monotony':         'Monotonia',
    'strain':           'Training Strain',
    'km_change_w0_w1':  'Cambio km (w0 vs w1)',
    'km_change_w1_w2':  'Cambio km (w1 vs w2)',
    'recovery_trend':   'Tendencia recuperacion',
    'exertion_trend':   'Tendencia esfuerzo',
}
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for ax, (col, label) in zip(axes, new_vars_labels.items()):
    data_0 = df_weekly[df_weekly['injury']==0][col].dropna()
    data_1 = df_weekly[df_weekly['injury']==1][col].dropna()
    bp = ax.boxplot([data_0, data_1], patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3),
                    whiskerprops=dict(linewidth=1.2), boxprops=dict(linewidth=1.2))
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Sin lesion', 'Lesion'], fontsize=9)
    ax.set_title(label, fontsize=10)
plt.suptitle('Variables derivadas por clase de lesion', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig07_features_derivadas.png'), bbox_inches='tight')
plt.show()
print("Bloque 2 completado.")

---
## Bloque 3 — Preparación para modelado

Este bloque realiza tres operaciones fundamentales antes de entrenar los modelos:

1. **Selección de features** — 38 variables organizadas en 4 grupos
2. **Imputación de NaN** — mediana por atleta para respetar diferencias individuales  
3. **Split temporal 70/30** — los primeros 70% del periodo como train, el último 30% como test

> ⚠️ **Por qué no usamos un split aleatorio:** con datos temporales, un split aleatorio
> implicaría entrenar con datos del futuro para predecir el pasado. La validación temporal
> simula el uso real del modelo.

In [ ]:
# ── Selección de features ─────────────────────────────────────────────────────
BASE_FEATURES = [
    'nr. sessions', 'nr. rest days', 'total kms', 'max km one day',
    'total km Z3-Z4-Z5-T1-T2', 'nr. tough sessions (effort in Z5, T1 or T2)',
    'nr. days with interval session', 'total km Z3-4', 'total km Z5-T1-T2',
    'total hours alternative training', 'nr. strength trainings',
    'avg exertion', 'min exertion', 'max exertion',
    'avg training success', 'min training success', 'max training success',
    'avg recovery', 'min recovery', 'max recovery',
]
LAG1_FEATURES = [
    'total kms.1', 'avg exertion.1', 'avg recovery.1',
    'nr. sessions.1', 'nr. rest days.1',
    'nr. tough sessions (effort in Z5, T1 or T2).1',
    'total km Z5-T1-T2.1',
]
LAG2_FEATURES = [
    'total kms.2', 'avg exertion.2', 'avg recovery.2',
]
DERIVED_FEATURES = [
    'ACWR', 'ACWR_exertion', 'monotony', 'strain',
    'km_change_w0_w1', 'km_change_w1_w2',
    'recovery_trend', 'exertion_trend',
]
ALL_FEATURES = BASE_FEATURES + LAG1_FEATURES + LAG2_FEATURES + DERIVED_FEATURES
print(f"Total features seleccionadas: {len(ALL_FEATURES)}")
print(f"  Base: {len(BASE_FEATURES)} | Lag-1: {len(LAG1_FEATURES)} | "
      f"Lag-2: {len(LAG2_FEATURES)} | Derivadas: {len(DERIVED_FEATURES)}")

In [ ]:
# ── Imputacion de NaN ─────────────────────────────────────────────────────────
print(f"NaN antes de imputacion: {df_weekly[ALL_FEATURES].isnull().sum().sum():,}")

for feat in DERIVED_FEATURES:
    if df_weekly[feat].isna().sum() > 0:
        df_weekly[feat] = df_weekly.groupby('Athlete ID')[feat].transform(
            lambda x: x.fillna(x.median())
        )
        df_weekly[feat] = df_weekly[feat].fillna(df_weekly[feat].median())

print(f"NaN despues de imputacion: {df_weekly[ALL_FEATURES].isnull().sum().sum():,}")

In [ ]:
# ── Split temporal 70/30 ─────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

SPLIT_DATE = df_weekly['Date'].quantile(0.70)

train_mask = df_weekly['Date'] <= SPLIT_DATE
test_mask  = df_weekly['Date'] >  SPLIT_DATE

X_train = df_weekly.loc[train_mask, ALL_FEATURES].copy()
X_test  = df_weekly.loc[test_mask,  ALL_FEATURES].copy()
y_train = df_weekly.loc[train_mask, 'injury'].copy()
y_test  = df_weekly.loc[test_mask,  'injury'].copy()

# Estandarizacion (fit solo sobre train para evitar data leakage)
scaler       = StandardScaler()
X_train_sc   = pd.DataFrame(scaler.fit_transform(X_train),
                             columns=ALL_FEATURES, index=X_train.index)
X_test_sc    = pd.DataFrame(scaler.transform(X_test),
                             columns=ALL_FEATURES, index=X_test.index)

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Split temporal (cutoff Date={SPLIT_DATE:.0f})")
print(f"  Train: {len(X_train):,} obs. | lesiones: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"  Test:  {len(X_test):,} obs.  | lesiones: {y_test.sum()} ({y_test.mean()*100:.2f}%)")
print(f"  Ratio desbalanceo train: 1:{SCALE_POS_WEIGHT:.0f}")
print(f"  scale_pos_weight XGBoost: {SCALE_POS_WEIGHT:.1f}")

In [ ]:
# ── Figura 9: Esquema del split temporal ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

obs_by_date = df_weekly.groupby('Date').size()
axes[0].fill_between(obs_by_date.index, obs_by_date.values,
                     where=obs_by_date.index <= SPLIT_DATE,
                     alpha=0.6, color='#2196F3', label='Train (70%)')
axes[0].fill_between(obs_by_date.index, obs_by_date.values,
                     where=obs_by_date.index > SPLIT_DATE,
                     alpha=0.6, color='#FF9800', label='Test (30%)')
axes[0].axvline(SPLIT_DATE, color='black', linestyle='--', linewidth=1.5)
axes[0].set_ylabel('Observaciones / dia')
axes[0].set_title('Distribucion temporal y split train/test')
axes[0].legend(loc='upper left')

inj_by_date = df_weekly.groupby('Date')['injury'].mean() * 100
smooth = inj_by_date.rolling(window=30, min_periods=1).mean()
axes[1].plot(smooth.index, smooth.values, color='#F44336', linewidth=1.5)
axes[1].fill_between(smooth.index, smooth.values,
                     where=smooth.index <= SPLIT_DATE, alpha=0.15, color='#2196F3')
axes[1].fill_between(smooth.index, smooth.values,
                     where=smooth.index > SPLIT_DATE, alpha=0.15, color='#FF9800')
axes[1].axvline(SPLIT_DATE, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Indice temporal (dias)')
axes[1].set_ylabel('Tasa de lesion (%) [media movil 30d]')
axes[1].set_title('Tasa de lesion a lo largo del tiempo')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig09_split_temporal.png'), bbox_inches='tight')
plt.show()
print("Bloque 3 completado. Listo para modelado.")

---
## Bloque 4 — Modelos predictivos

En este bloque entrenamos y evaluamos tres algoritmos de clasificación supervisada para predecir
la variable `injury`. La selección de modelos responde a criterios de complementariedad:

| Modelo | Justificación |
|--------|---------------|
| **Regresión Logística** | Modelo lineal interpretable; actúa como baseline. Uso de `class_weight='balanced'` para compensar el desbalanceo. |
| **Random Forest** | Ensemble no lineal basado en bagging; captura interacciones sin requerir ingeniería manual. Mismo ajuste por `class_weight`. |
| **XGBoost** | Gradient boosting con regularización; permite ajustar `scale_pos_weight` para el desbalanceo extremo (~85:1). |

> ⚠️ **Nota sobre la evaluación:** Dado el desbalanceo severo (~1.2% de lesiones), el accuracy
> no es una métrica informativa (un modelo trivial que prediga siempre 0 alcanzaría ~98.8%).
> La evaluación se centra en **ROC-AUC**, **F1**, **precision** y **recall** sobre la clase
> minoritaria.

In [ ]:
# ── Imports Bloque 4 ──────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import time

print("Librerias de modelado cargadas correctamente.")

### 4.1 Entrenamiento de los tres modelos

Cada modelo se entrena sobre `X_train_sc` (datos estandarizados) y `y_train`, y se evalúa
sobre el conjunto de test. Se registran las probabilidades predichas para calcular ROC-AUC
y las curvas correspondientes.

In [ ]:
# ── Definición y entrenamiento ─────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=SCALE_POS_WEIGHT,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ),
}

results = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_sc, y_train)
    elapsed = time.time() - t0
    
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    
    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'avg_prec':  average_precision_score(y_test, y_proba),
        'time':      elapsed,
    }
    print(f"✓ {name:25s} entrenado en {elapsed:.1f}s  |  "
          f"ROC-AUC={results[name]['roc_auc']:.4f}  "
          f"F1={results[name]['f1']:.4f}  "
          f"Recall={results[name]['recall']:.4f}")

### 4.2 Tabla comparativa de rendimiento

In [ ]:
# ── Tabla resumen ──────────────────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    name: {
        'ROC-AUC':           r['roc_auc'],
        'Average Precision':  r['avg_prec'],
        'F1 (clase 1)':      r['f1'],
        'Precision (clase 1)': r['precision'],
        'Recall (clase 1)':  r['recall'],
    }
    for name, r in results.items()
}).T

metrics_df = metrics_df.round(4)
print("\n" + "=" * 75)
print("COMPARATIVA DE RENDIMIENTO — CONJUNTO DE TEST")
print("=" * 75)
print(metrics_df.to_string())
print("=" * 75)

# Mejor modelo por ROC-AUC
best_name = metrics_df['ROC-AUC'].idxmax()
print(f"\nMejor modelo por ROC-AUC: {best_name} ({metrics_df.loc[best_name, 'ROC-AUC']:.4f})")

### 4.3 Matrices de confusión

In [ ]:
# ── Figura 10: Matrices de confusión ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    # Normalizar por fila para mostrar tasas
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No lesion', 'Lesion'],
                yticklabels=['No lesion', 'Lesion'],
                cbar=False, linewidths=0.5)
    
    # Añadir porcentajes
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f"({cm_norm[i, j]:.1%})",
                    ha='center', va='center', fontsize=8, color='gray')
    
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Real' if ax == axes[0] else '')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de confusión — Conjunto de test', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig10_confusion_matrices.png'), bbox_inches='tight')
plt.show()

### 4.4 Curvas ROC y Precision-Recall

En problemas con desbalanceo severo, la curva ROC puede ser optimista porque la clase
mayoritaria domina la tasa de falsos positivos. Por eso complementamos con la curva
Precision-Recall, que evalúa directamente el rendimiento sobre la clase minoritaria.

In [ ]:
# ── Figura 11: Curvas ROC y Precision-Recall ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = {'Logistic Regression': '#2196F3', 'Random Forest': '#4CAF50', 'XGBoost': '#FF9800'}

# Curva ROC
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax1.plot(fpr, tpr, color=colors[name], linewidth=2,
             label=f"{name} (AUC={r['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Aleatorio')
ax1.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax1.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax1.set_title('Curva ROC')
ax1.legend(loc='lower right', fontsize=9)
ax1.set_xlim([-0.02, 1.02])
ax1.set_ylim([-0.02, 1.02])

# Curva Precision-Recall
baseline_pr = y_test.mean()
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r['y_proba'])
    ax2.plot(rec, prec, color=colors[name], linewidth=2,
             label=f"{name} (AP={r['avg_prec']:.3f})")
ax2.axhline(baseline_pr, color='k', linestyle='--', linewidth=0.8, alpha=0.5,
            label=f'Baseline ({baseline_pr:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Curva Precision-Recall')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_xlim([-0.02, 1.02])
ax2.set_ylim([0, max(0.15, baseline_pr * 5)])

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig11_roc_pr_curves.png'), bbox_inches='tight')
plt.show()

### 4.5 Importancia de variables

Analizamos qué variables contribuyen más a las predicciones de cada modelo:
- **Regresión Logística**: coeficientes estandarizados (valor absoluto).
- **Random Forest**: importancia por impureza (Gini).
- **XGBoost**: importancia por ganancia (gain).

In [ ]:
# ── Importancia de variables por modelo ────────────────────────────────────────
TOP_N = 15

# Logistic Regression — coeficientes absolutos
lr_coefs = pd.Series(
    np.abs(results['Logistic Regression']['model'].coef_[0]),
    index=ALL_FEATURES
).sort_values(ascending=False)

# Random Forest — importancia Gini
rf_imp = pd.Series(
    results['Random Forest']['model'].feature_importances_,
    index=ALL_FEATURES
).sort_values(ascending=False)

# XGBoost — importancia por ganancia
xgb_imp = pd.Series(
    results['XGBoost']['model'].feature_importances_,
    index=ALL_FEATURES
).sort_values(ascending=False)

# ── Figura 12: Top features por modelo ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (imp, title, color) in zip(axes, [
    (lr_coefs.head(TOP_N),  'Logistic Regression\n(|coeficientes|)', '#2196F3'),
    (rf_imp.head(TOP_N),    'Random Forest\n(Gini importance)',       '#4CAF50'),
    (xgb_imp.head(TOP_N),   'XGBoost\n(Gain importance)',             '#FF9800'),
]):
    imp_sorted = imp.sort_values(ascending=True)
    ax.barh(range(len(imp_sorted)), imp_sorted.values, color=color, alpha=0.8)
    ax.set_yticks(range(len(imp_sorted)))
    ax.set_yticklabels(imp_sorted.index, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Importancia')

plt.suptitle(f'Top {TOP_N} variables más importantes por modelo', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig12_feature_importance.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Ranking de consenso ────────────────────────────────────────────────────────
# Normalizamos cada ranking a [0,1] y promediamos posiciones
def rank_normalize(series):
    ranked = series.rank(ascending=False)
    return 1 - (ranked - 1) / (len(ranked) - 1)

consensus = pd.DataFrame({
    'LR':  rank_normalize(lr_coefs),
    'RF':  rank_normalize(rf_imp),
    'XGB': rank_normalize(xgb_imp),
})
consensus['Media'] = consensus.mean(axis=1)
consensus = consensus.sort_values('Media', ascending=False)

print("TOP 15 VARIABLES — RANKING DE CONSENSO (3 modelos)")
print("=" * 65)
print(consensus.head(15).round(3).to_string())
print("=" * 65)

### 4.6 Análisis de sensibilidad al umbral de decisión

El umbral por defecto (0.5) no es óptimo en problemas con desbalanceo severo. Evaluamos
cómo varía el F1 del mejor modelo en función del umbral para identificar el punto de
equilibrio entre precision y recall.

In [ ]:
# ── Figura 13: Sensibilidad al umbral ─────────────────────────────────────────
best_name = metrics_df['ROC-AUC'].idxmax()
best_proba = results[best_name]['y_proba']

thresholds = np.arange(0.01, 0.50, 0.005)
f1s, precs, recs = [], [], []

for t in thresholds:
    y_t = (best_proba >= t).astype(int)
    f1s.append(f1_score(y_test, y_t, zero_division=0))
    precs.append(precision_score(y_test, y_t, zero_division=0))
    recs.append(recall_score(y_test, y_t, zero_division=0))

optimal_idx = np.argmax(f1s)
optimal_threshold = thresholds[optimal_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,  color='#F44336', linewidth=2, label='F1')
ax.plot(thresholds, precs, color='#2196F3', linewidth=1.5, linestyle='--', label='Precision')
ax.plot(thresholds, recs,  color='#4CAF50', linewidth=1.5, linestyle='--', label='Recall')
ax.axvline(optimal_threshold, color='black', linestyle=':', linewidth=1,
           label=f'Umbral optimo = {optimal_threshold:.3f}')
ax.axvline(0.5, color='gray', linestyle=':', linewidth=0.8, alpha=0.5, label='Umbral default (0.5)')
ax.set_xlabel('Umbral de decision')
ax.set_ylabel('Metrica')
ax.set_title(f'Sensibilidad al umbral — {best_name}')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig13_threshold_analysis.png'), bbox_inches='tight')
plt.show()

print(f"Umbral optimo (max F1): {optimal_threshold:.3f}")
print(f"  F1 = {f1s[optimal_idx]:.4f}  |  "
      f"Precision = {precs[optimal_idx]:.4f}  |  "
      f"Recall = {recs[optimal_idx]:.4f}")

# Comparativa default vs óptimo
print(f"\nComparativa con umbral default (0.5):")
print(f"  F1 default:  {results[best_name]['f1']:.4f}")
print(f"  F1 optimo:   {f1s[optimal_idx]:.4f} ({(f1s[optimal_idx]/results[best_name]['f1']-1)*100:+.1f}%)")

### 4.7 Análisis de errores

Para entender dónde fallan los modelos, analizamos los falsos negativos (lesiones no
detectadas) y los falsos positivos (falsas alarmas). Esto es esencial para valorar la
utilidad práctica del sistema predictivo.

In [ ]:
# ── Análisis de errores del mejor modelo ──────────────────────────────────────
best_pred  = results[best_name]['y_pred']
best_proba_arr = results[best_name]['y_proba']

# Clasificación de errores
test_analysis = pd.DataFrame({
    'y_real':  y_test.values,
    'y_pred':  best_pred,
    'proba':   best_proba_arr,
}, index=X_test.index)

for feat in ['ACWR', 'total kms', 'avg recovery', 'avg exertion', 'strain', 'monotony']:
    test_analysis[feat] = df_weekly.loc[X_test.index, feat]

test_analysis['tipo'] = 'TN'
test_analysis.loc[(test_analysis['y_real']==1) & (test_analysis['y_pred']==1), 'tipo'] = 'TP'
test_analysis.loc[(test_analysis['y_real']==1) & (test_analysis['y_pred']==0), 'tipo'] = 'FN'
test_analysis.loc[(test_analysis['y_real']==0) & (test_analysis['y_pred']==1), 'tipo'] = 'FP'

print(f"Distribucion de predicciones ({best_name}):")
print(test_analysis['tipo'].value_counts().to_string())

# ── Figura 14: Perfil de errores ──────────────────────────────────────────────
error_vars = ['ACWR', 'total kms', 'avg recovery', 'avg exertion', 'strain']
fig, axes = plt.subplots(1, len(error_vars), figsize=(16, 4))

tipo_colors = {'TP': '#4CAF50', 'FN': '#F44336', 'FP': '#FF9800', 'TN': '#BDBDBD'}
tipo_order = ['TN', 'FP', 'FN', 'TP']

for ax, var in zip(axes, error_vars):
    data_plot = [test_analysis.loc[test_analysis['tipo'] == t, var].dropna() for t in tipo_order]
    bp = ax.boxplot(data_plot, labels=tipo_order, patch_artist=True, widths=0.6,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch, t in zip(bp['boxes'], tipo_order):
        patch.set_facecolor(tipo_colors[t])
        patch.set_alpha(0.7)
    ax.set_title(var, fontsize=10)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle(f'Perfil de variables por tipo de prediccion — {best_name}',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'fig14_error_analysis.png'), bbox_inches='tight')
plt.show()

# Estadísticas FN vs TP
print("\nPerfil medio — Falsos Negativos vs Verdaderos Positivos:")
print("-" * 55)
for var in error_vars:
    fn_mean = test_analysis.loc[test_analysis['tipo']=='FN', var].mean()
    tp_mean = test_analysis.loc[test_analysis['tipo']=='TP', var].mean()
    print(f"  {var:<20s}  FN={fn_mean:.3f}  TP={tp_mean:.3f}")

### 4.8 Resumen del Bloque 4

Los tres modelos han sido entrenados y evaluados sobre el conjunto de test con métricas
apropiadas para el contexto de desbalanceo. Los resultados principales se discutirán en
profundidad en el **Bloque 5 — Discusión**.

**Figuras generadas en este bloque:**
- Figura 10: Matrices de confusión
- Figura 11: Curvas ROC y Precision-Recall
- Figura 12: Importancia de variables (top 15 por modelo)
- Figura 13: Sensibilidad al umbral de decisión
- Figura 14: Perfil de errores por tipo de predicción